# BGS+: preprocessing

Prepares BGS+ for the experiments of Milestone 3, with the same steps as `dmg777k_preprocessing.ipynb`, so the two can be read side by side. The authors have no pipeline for BGS+. `data/bgs_data_prep.py` mirrors theirs for dmg777k and also converts British National Grid metres to WGS84. This notebook adds the same fixes as for dmg777k. It makes no modelling decisions; those are listed at the end.

**To run it:** prepare the data as in `data/README.md`, then run this notebook from top to bottom (about 1 minute). The same steps run without the notebook with `uv run python -m preprocessing.pipeline bgs`.

In [1]:
import pandas as pd

from preprocessing import pipeline
from preprocessing.geometry import geodesic_km

pd.set_option("display.max_colwidth", 70, "display.width", 160)


def short(iri: str) -> str:
    """Last part of an IRI, for display only; the saved tables keep full IRIs."""
    return iri.rstrip("/").rsplit("/", 1)[-1].rsplit("#", 1)[-1]


r = pipeline.run("bgs")
print(f"{len(r.dataset.triples):,} triples loaded, {len(r.triples):,} kept")

916,199 triples loaded, 882,514 kept


## 1. Preparing the data

### 1.1 The pipeline, with fixes

`data/bgs_data_prep.py` has already parsed the N-Triples, kept each object's type and reprojected every geometry to WGS84. Here we make the same four changes as for dmg777k:

- **Missing values are removed**, using the same list of placeholder texts.
- **A node is identified by its text and its kind.**
- **A geometry is any literal typed as a WKT literal.**
- **Coordinates are always (longitude, latitude).**

**Missing values.** Triples whose object is a placeholder text, grouped by that text:

In [2]:
removed = r.placeholders_removed.assign(predicate=r.placeholders_removed["predicate"].map(short))
print(f"Triples removed: {removed['triples'].sum():,}")
removed.groupby("object")["triples"].sum().sort_values(ascending=False).to_frame()

Triples removed: 33,685


,triples
object,
Not Entered,33139
Not defined.,396
Unknown.,44
Not known.,39
Not applicable.,23
N/a,7
Not known,7
Unknown,6
Not Applicable,6


**Interpretation:** BGS+ marks missing values with words rather than leaving them empty. "Not Entered" alone fills 33,139 triples, almost all in `isLocallyDominant` and `hasRoleDescription`. In the authors' graph each such word would be one node shared by tens of thousands of unrelated entities: a hub with no meaning. We remove all of these placeholders; the list matches them in any letter case, with or without a final full stop. Placeholders that BGS+ models as **entities** (for example `ShapeType/NotEntered`) are real nodes, so they stay. They appear in the hub list below.

**Node identity.** The graph with nodes identified by text (the authors) and by text and kind (ours):

In [3]:
r.identity

,By text (authors),By text and kind (ours)
Nodes,273946,333825
Self-loops,58689,0
Nodes where a literal is merged into an entity,58709,0
Nodes where literals of different types are merged,1161,0


**Interpretation:** identifying nodes by text alone merges about 60,000 nodes, far more than in dmg777k:

- **58,709 entities are merged with a literal**: their `identifier`, which holds the same text as their IRI. This gives 58,689 self-loops, and every labelled rock unit has one.
- **1,161 texts merge literals of different types**, such as the numbers 0, 1 and 2 typed as integer, int and decimal.

With nodes identified by text and kind, there are no self-loops and no wrong merges.

**Geometries.** WKT literals, and how many hold a usable geometry:

In [4]:
print(r.geometry_counts.to_dict())
print("Geometry types:", r.geometries["geometry_type"].value_counts().to_dict())
print(f"Entities with their own geometry: {len(r.locations):,}")

{'WKT literals': 48604, 'Not valid WKT': 0, 'Empty': 1, 'Usable': 48603}
Geometry types: {'Polygon': 23917, 'Point': 21943, 'LineString': 2743}
Entities with their own geometry: 22,784


**Interpretation:** 48,603 of the 48,604 WKT literals are usable. The one left out is `POLYGON EMPTY`, used by two stratotypes, which has no location. 22,784 entities have their own geometry: map units, dykes, faults and stratotypes. As the EDA showed, no labelled rock unit has a geometry of its own; they get one only through flooding.

### 1.2 Checks: the problems from the EDA are gone

Each row is a problem the EDA found. The first column shows the authors' pipeline, the second ours. The distance row tests the coordinate handling itself, on two Dutch points with a known distance (geopy gives 34.6 km), so it reads the same as for dmg777k.

In [5]:
r.checks

,authors' pipeline,ours,expected
check,,,
WKT literals recognised as geometries,48603.0,48603.0,"all usable of 48,604"
Geometry centres outside the country,NaN,0.0,0
"Amsterdam–Utrecht distance, km (true 34.6)",39.4,34.6,34.6
Self-loops,58689.0,0.0,0
Nodes merging an entity and a literal,58709.0,0.0,0
Nodes merging literals of different types,1161.0,0.0,0


**Interpretation:**

- **The authors' expression finds every geometry here**, since BGS+ always writes the type in capitals.
- **All centres lie in the British Isles**, so the reprojection and the coordinate order are right.
- **There are no self-loops and no wrong merges left.**

## 2. Describing the relations

The same table as for dmg777k: number of edges, distinct objects, whether it points to entities or literals, and the spatial spread of the entities that share an object. Relations with fewer than 30 such entities get no value. The table holds facts only; which relations pass on locations is decided in the experiments.

In [6]:
loc = r.locations
baseline_km = pd.Series(geodesic_km(loc["lon"], loc["lat"], loc["lon"].mean(), loc["lat"].mean())).median()
print(f"Baseline: median distance of a located entity to the mean position of all of them: {baseline_km:.1f} km")
relations = r.relations.assign(relation=r.relations["relation"].map(short), literal_kind=r.relations["literal_kind"].map(lambda k: k if pd.isna(k) else short(k)))
print(f"{len(relations)} relations, {relations['median_km'].notna().sum()} with a spatial spread")
relations.dropna(subset=["median_km"]).sort_values("median_km")

Baseline: median distance of a located entity to the mean position of all of them: 248.4 km
103 relations, 20 with a spatial spread


,relation,edges,distinct_objects,object,literal_kind,located_subjects,median_km
8,hasPointLocation,22784,21943,literal,wktLiteral,1348.0,0.00
9,hasBoundingBox,22784,21945,literal,XMLLiteral,1344.0,0.00
11,spatial,22784,21959,literal,wktLiteral,1316.0,0.00
10,hasFootprint,22784,21959,literal,wktLiteral,1316.0,0.00
44,hasLocationDescription,5533,5303,literal,@en,358.0,0.08
43,isStratotypeFor,5534,3486,entity,NaN,3312.0,4.05
1,label,58966,58790,literal,@en,43.0,33.91
14,prefLabel,16052,15943,literal,@en,43.0,33.91
18,hasLexicon,14507,154,entity,NaN,14499.0,85.27
15,hasMapCode,14507,110,literal,plain,14506.0,106.01


**Interpretation:** against a baseline of 248 km, few relations are spatial:

- **Spatial:** the geometry relations themselves, `hasLocationDescription` (0.08 km) and `isStratotypeFor` (4 km).
- **Weakly spatial:** `hasLexicon`, `hasMapCode` and `hasColourRGB` (85–112 km). Map units showing the same rock type spread across a region.
- **Not spatial:** ages, ranks, `type`, `inScheme` and `isDefinedBy` (117–248 km).

`hasBoundingBox` points to WKT literals for some entities and to GML text (`XMLLiteral`) for others. Only the WKT ones count as geometries. The GML boxes stay as ordinary literals. All 17,250 entities that have one also have a WKT footprint and point, so no location is lost.

The full table is saved as `relations.csv`:

In [7]:
relations

,relation,edges,distinct_objects,object,literal_kind,located_subjects,median_km
0,type,83523,6,entity,NaN,22784.0,248.42
1,label,58966,58790,literal,@en,43.0,33.91
2,identifier,58738,58738,literal,@en,NaN,NaN
3,inScheme,34956,25,entity,NaN,17250.0,227.16
4,isDefinedBy,34550,24,entity,NaN,17250.0,227.16
...,...,...,...,...,...,...,...
98,hasSourceOfName,1,1,literal,@en,NaN,NaN
99,hasRoleDescription,1,1,literal,@en,NaN,NaN
100,about,1,1,literal,anyURI,NaN,NaN
101,hasTopConcept,1,1,entity,NaN,NaN,NaN


### Hubs

The nodes with the most edges, and the relation that brings most of them. The EDA found 50.3% of BGS+ nodes at one national centroid after flooding, because of hubs like these.

In [8]:
hubs = r.hubs.assign(node=r.hubs["node"].str[-60:], kind=r.hubs["kind"].map(short), main_relation=r.hubs["main_relation"].map(short))
hubs.head(15)

,node,kind,degree,in_degree,out_degree,main_relation,main_relation_share
0,http://www.w3.org/2004/02/skos/core#Concept,entity,83024,83024,0,type,1.000
1,http://data.bgs.ac.uk/ref/Lexicon/NamedRockUnit,entity,23403,23394,9,inScheme,0.500
2,http://data.bgs.ac.uk/ref/625KGeologyMap/Unit,entity,22493,22488,5,isDefinedBy,0.500
3,http://www.w3.org/1999/02/22-rdf-syntax-ns#nil,entity,15235,15235,0,subClassUDCS,0.193
4,http://data.bgs.ac.uk/ref/CurrentStatus/CURRENT,entity,12759,12759,0,hasCurrentStatus,1.000
5,tp://data.bgs.ac.uk/id/EarthMaterialClass/ComponentRank/MAIN,entity,12729,12729,0,hasComponentRank,1.000
6,http://data.bgs.ac.uk/id/Lexicon/ShapeType/NotEntered,entity,11680,11675,5,hasShapeType,1.000
7,http://data.bgs.ac.uk/id/onshoreOffshore/ONSHORE,entity,11101,11101,0,isOnshoreOffshore,1.000
8,"""FORMAL""",plain,10559,10559,0,hasFormality,1.000
9,http://data.bgs.ac.uk/id/Lexicon/Theme/BEDRCK,entity,10026,10020,6,hasTheme,0.999


**Interpretation:** `skos:Concept` is the largest hub (83,024 edges, all through `type`). It is followed by the scheme nodes (`NamedRockUnit`, `625KGeologyMap/Unit`), `rdf:nil` (the end of every RDF list) and status values such as `CURRENT` and `ONSHORE`. Some hubs are placeholder entities, such as `ShapeType/NotEntered` and `SpatialScope/NotApplicable`. None of them is a place. The list is saved as `hubs.csv`, for the experiments to decide what to do with them.

## 3. Labels and splits

The published train, validation and test files. Every check should be 0.

In [9]:
display(r.splits)
r.label_checks

split,train,valid,test,total
class,,,,
FLUV,59,15,19,93
GLACI,35,8,10,53
total,94,23,29,146


,count
Nodes labelled more than once,0
Labelled nodes missing from the graph,0
Triples using the target relation (hasLithogenesis),0


**Interpretation:** 146 rock units, 93 FLUV and 53 GLACI, with the same class shares in each split. No unit appears twice, all are in the graph, and the target relation `hasLithogenesis` is absent, so no label leaks into the graph.

## 4. Saving

All outputs go to `data/bgs/processed/` (git-ignored), in the same format as for dmg777k:

| File | Contents |
|---|---|
| `triples.parquet` | subject, predicate, object, object_kind, with node keys as in dmg777k |
| `geometries.parquet` | node key, WKT, geometry type and centre (lon, lat) of every usable WKT literal |
| `labels.csv` | node, class, split |
| `relations.csv` | the relation table of Section 2 |
| `hubs.csv` | the 50 nodes with the most edges |

In [10]:
pipeline.save(r)
for path in sorted(r.dataset.processed_dir.iterdir()):
    print(f"{path.name:20} {path.stat().st_size / 1e6:8.1f} MB")

geometries.parquet       26.7 MB
hubs.csv                  0.0 MB
labels.csv                0.0 MB
relations.csv             0.0 MB
triples.parquet          50.7 MB


## What is left to the experiments

Preprocessing makes no modelling decisions. These belong to the experiments (see `experiments/README.md`):

- **Walk direction and walk start.** Triples are stored in their original direction. Whether walks may also follow edges backwards, and which nodes they start from, are walker settings.
- **Which relations pass on a location during flooding, and what to do with hubs.** The relation table and the hub list above are the facts these choices are based on. The choices themselves live in experiment configurations.
- **Flooding and edge weights.** `geometries.parquet` holds the starting geometries. Flooding them through the graph is the experiment.
